In [1]:
# ===============================
# Jupyter bootstrap cell
# Run this after VS Code Web restart
# ===============================

import sys
import subprocess
import importlib

def ensure_package(pkg):
    try:
        importlib.import_module(pkg)
        print(f"✅ {pkg} already installed")
    except ImportError:
        print(f"📦 Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# --- Ensure required packages ---
ensure_package("google-cloud-bigquery")
ensure_package("pandas")
ensure_package("pyarrow")
ensure_package("db-dtypes")

# --- Restart imports cleanly ---
from google.cloud import bigquery
import pandas as pd

# --- Initialize BigQuery client ---
client = bigquery.Client()

print("✅ BigQuery client initialized")
print("Project:", client.project)


📦 Installing google-cloud-bigquery ...
  Using cached google_cloud_bigquery-3.40.0-py3-none-any.whl.metadata (8.2 kB)
Using cached google_cloud_bigquery-3.40.0-py3-none-any.whl (261 kB)
✅ pandas already installed
✅ pyarrow already installed
📦 Installing db-dtypes ...
  Using cached db_dtypes-1.5.0-py3-none-any.whl.metadata (3.4 kB)
Using cached db_dtypes-1.5.0-py3-none-any.whl (18 kB)
✅ BigQuery client initialized
Project: ai-agentic-bootcamp-july-2025


In [2]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client()

query = """
SELECT *
FROM `project-ad8e168b-9904-43b7-b43.Vector_AI_POC.Health_Insurance_Synthetic`
"""

job = client.query(
    query,
    project="project-ad8e168b-9904-43b7-b43"  # job runs in your personal project
)

df = job.to_dataframe()

print(df.shape)


/home/coder/agent-bootcamp/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(1338, 10)


In [3]:
df.head()

,age,sex,bmi,children,smoker,region,charges,systolic_bp,diastolic_bp,hba1c
0,18,female,26.315,0,False,northeast,2198.18985,108.0,84.0,5.90
1,18,female,38.665,2,False,northeast,3393.35635,108.0,77.0,5.88
2,18,female,35.625,0,False,northeast,2211.13075,130.0,79.0,5.20
3,18,female,30.115,0,False,northeast,21344.84670,152.0,89.0,5.90
4,18,male,23.750,0,False,northeast,1705.62450,115.0,78.0,6.21


In [4]:
# Test to run schema Agent
# python - <<EOF
# from agents.schema_agent import load_table_schema

# schema = load_table_schema(
#     "project-ad8e168b-9904-43b7-b43.Vector_AI_POC.Health_Insurance_Synthetic"
# )

# print("TABLE:", schema.table_name)
# print("COLUMN COUNT:", len(schema.columns))
# print("SAMPLE COLUMNS:", list(schema.columns.items())[:5])
# EOF


In [5]:
# Test to run intent agent
# python - <<EOF
# from agents.planner_agent import create_execution_plan

# plan = create_execution_plan(
#     user_query="Show average premium by gender",
#     available_columns=["premium", "gender", "age"]
# )

# print(plan)
# EOF




In [6]:
from agents.schema_agent import load_table_schema
from chatbot.agents.intent_planner_agent import parse_intent

schema = load_table_schema(
    "project-ad8e168b-9904-43b7-b43.Vector_AI_POC.Health_Insurance_Synthetic"
)

def run_test(query: str):
    print("=" * 80)
    print("QUERY:", query)
    intent = parse_intent(query, schema)
    print(intent)

# -----------------------------
# Test cases
# -----------------------------

run_test("Show average age by sex")

run_test("How many smokers are there?")

run_test("List the cost per region")

run_test("Plot average bmi by age")

run_test("Give me the total number of rows in the table")


ModuleNotFoundError: No module named 'chatbot'

In [7]:
# Test intent and planner agent 
from agents.schema_agent import load_table_schema
from agents.intent_planner_agent import run_intent_planner_agent

schema = load_table_schema("project-ad8e168b-9904-43b7-b43.Vector_AI_POC.Health_Insurance_Synthetic")
plan, intent = run_intent_planner_agent("Show average age by sex", schema)

print("PLAN:", plan)
print("INTENT:", intent)

PLAN: ExecutionPlan(steps=['aggregate', 'group_by', 'visualize'], output_format=<OutputFormat.TABLE: 'table'>, chart_type=None, needs_clarification=False, clarification_question=None, is_valid=True)
INTENT: ParsedIntent(metrics=['age'], filters={}, group_by='sex', output_format=<OutputFormat.TABLE: 'table'>, requires_aggregation=True, is_valid=True, clarification_question=None)


In [8]:
# Test SQL planner agent 
from agents.schema_agent import load_table_schema
from agents.intent_planner_agent import run_intent_planner_agent
from agents.sql_planner_agent import generate_sql

TABLE = "project-ad8e168b-9904-43b7-b43.Vector_AI_POC.Health_Insurance_Synthetic"

schema = load_table_schema(TABLE)
plan, intent = run_intent_planner_agent("Show average age by sex", schema)

sqlq = generate_sql(intent=intent, schema=schema, table_name=TABLE)
print("SQL SAFE:", sqlq.is_safe)
print("ERRORS:", sqlq.validation_errors)
print("SQL:\\n", sqlq.query)



SQL SAFE: False
ERRORS: ['What calculation would you like to perform on age? For example, average, minimum, maximum, or sum?']
SQL:\n 


In [10]:
# Test till the SQl Agent Planner
from agents.schema_agent import load_table_schema
from agents.intent_planner_agent import run_intent_planner_agent
from agents.sql_planner_agent import generate_sql

# -----------------------------
# CONFIG
# -----------------------------
TABLE = "project-ad8e168b-9904-43b7-b43.Vector_AI_POC.Health_Insurance_Synthetic"

TEST_QUERIES = [
    "Show average age by sex",
    "How many smokers are there?",
    "List the number of rows by region",
    "Plot average bmi by age",
]

# -----------------------------
# TEST START
# -----------------------------
print("=== LOADING SCHEMA ===")
schema = load_table_schema(TABLE)
print("Table:", schema.table_name)
print("Columns:", list(schema.columns.keys()))
print("")

for q in TEST_QUERIES:
    print("=" * 80)
    print("QUERY:", q)

    # 1️⃣ Intent + Planning Agent
    plan, intent = run_intent_planner_agent(q, schema)

    print("\nPLAN:")
    print(plan)

    print("\nINTENT:")
    print(intent)

    assert plan.is_valid, "ExecutionPlan is invalid"
    assert intent.is_valid, "ParsedIntent is invalid"

    # 2️⃣ SQL Planner Agent
    sqlq = generate_sql(
        intent=intent,
        schema=schema,
        table_name=TABLE,
    )

    print("\nSQL SAFE:", sqlq.is_safe)
    if not sqlq.is_safe:
        print("SQL ERRORS:", sqlq.validation_errors)

    print("\nSQL QUERY:")
    print(sqlq.query)

    assert sqlq.is_safe, "Generated SQL is unsafe"

print("\n✅ ALL AGENT TESTS PASSED (UP TO SQL PLANNER)")



=== LOADING SCHEMA ===
Table: project-ad8e168b-9904-43b7-b43.Vector_AI_POC.Health_Insurance_Synthetic
Columns: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges', 'systolic_bp', 'diastolic_bp', 'hba1c']

QUERY: Show average age by sex

PLAN:
ExecutionPlan(steps=['aggregate', 'group_by', 'visualize'], output_format=<OutputFormat.TABLE: 'table'>, chart_type=None, needs_clarification=False, clarification_question=None, is_valid=True)

INTENT:
ParsedIntent(metrics=['age'], filters={}, group_by='sex', output_format=<OutputFormat.TABLE: 'table'>, requires_aggregation=True, is_valid=True, clarification_question=None)

SQL SAFE: True

SQL QUERY:
SELECT sex, AVG(age) AS average_age FROM `project-ad8e168b-9904-43b7-b43.Vector_AI_POC.Health_Insurance_Synthetic` GROUP BY sex
QUERY: How many smokers are there?

PLAN:
ExecutionPlan(steps=['aggregate'], output_format=<OutputFormat.VALUE: 'value'>, chart_type=None, needs_clarification=False, clarification_question=None, is_valid=True)

IN

In [18]:
import sys
sys.modules.pop("agents.execution_agent", None)


<module 'agents.execution_agent' from '/home/coder/agent-bootcamp/src/chatbot/agents/execution_agent.py'>

In [2]:
from agents.execution_agent import ExecutionAgent
from orchestrator.contracts import SQLQuery

SQL = """
SELECT sex, AVG(age) AS avg_age
FROM `project-ad8e168b-9904-43b7-b43.Vector_AI_POC.Health_Insurance_Synthetic`
GROUP BY sex
"""

agent = ExecutionAgent()
sqlq = SQLQuery(query=SQL, is_safe=True)

result = agent.execute(sqlq)

print("ROW COUNT:", result.row_count)
print("ROWS:", result.rows)


Forbidden: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/project-ad8e168b-9904-43b7-b43/jobs?prettyPrint=false: Caller does not have required permission to use project project-ad8e168b-9904-43b7-b43. Grant the caller the roles/serviceusage.serviceUsageConsumer role, or a custom role with the serviceusage.services.use permission, by visiting https://console.developers.google.com/iam-admin/iam/project?project=project-ad8e168b-9904-43b7-b43 and then retry. Propagation of the new permission may take a few minutes.

Location: None
Job ID: f251fdb7-0bf3-4856-9dd7-5b0f67f16cd8
 [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'USER_PROJECT_DENIED', 'domain': 'googleapis.com', 'metadata': {'containerInfo': 'project-ad8e168b-9904-43b7-b43', 'consoleUrl': 'https://console.developers.google.com/iam-admin/iam/project?project=project-ad8e168b-9904-43b7-b43', 'service': 'bigquery.googleapis.com', 'consumer': 'projects/project-ad8e168b-9904-43b7-b43'}}, {'@type': 'type